# H&M Dataset — FOAF Decentralized MF Experiment

Mirrors the ML-100K experiment structure exactly:
- Same FOAF gradient-sharing method (user-stratified, random-sampling batch)
- Same graph topologies: Random-2-out, Random-5-out, Scale-Free, Cycle
- Same evaluation metric: RMSE on held-out test set
- Same baselines: CL (centralized), FL (FedAvg), GL (Gossip), LL (local)

**Data source**: Download `articles.csv`, `customers.csv`, `transactions_train.csv`  
from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations

**Target**: purchase frequency `y_ij = count(customer_i, product_j)` — same as appendix.

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import datetime
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Dict, Optional
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Data Loading & Preprocessing (follows H_M_Data.ipynb)

In [5]:
# ── Load raw files ──────────────────────────────────────────────────────────
DATA_DIR = "."   # change to path containing the three CSVs

df           = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                           dtype={"article_id": str})
customer_df  = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                           dtype={"article_id": str, "product_code": str})
article_df   = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                           dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

# ── Week index (0 = most recent week) ───────────────────────────────────────
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"]  = (df["t_dat"].max() - df["t_dat"]).dt.days // 7

print("Raw transactions:", len(df))

FileNotFoundError: [Errno 2] No such file or directory: './transactions_train.csv'

In [ ]:
# ── Filter: top-6 product types, ≥120-purchase customers, top-5000 locations
# (exactly as in appendix Section 3.2 and H_M_Data.ipynb)
chosen_types = (
    article_df.groupby("product_type_name")["product_code"]
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby("customer_id")["product_code"].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df["postal_code"].value_counts()[1:5001].index
customer_df = customer_df[customer_df["postal_code"].isin(location_customers)]

chosen_articles = article_df[article_df["product_type_name"].isin(chosen_types)]
df = df[df["product_code"].isin(chosen_articles["product_code"])]
df = df[df["customer_id"].isin(chosen_customers)]
df = df[df["customer_id"].isin(customer_df["customer_id"])]

df.drop(["t_dat", "price", "article_id", "sales_channel_id"], axis=1, inplace=True)
df["bought"] = 1
df.reset_index(drop=True, inplace=True)

print("Filtered transactions:", len(df))

In [ ]:
# ── Temporal split: week > 50 → train history, week ≤ 50 → test
# (week=0 is most recent; lower week number = more recent)
TEST_WEEK_CUTOFF = 50   # most recent 50 weeks = test set

train_raw = df[df["week"] >  TEST_WEEK_CUTOFF].copy()   # older weeks = train
test_raw  = df[df["week"] <= TEST_WEEK_CUTOFF].copy()   # recent weeks = test

# Aggregate to (customer, product) → purchase count
train_df = (
    train_raw.groupby(["customer_id", "product_code"])["bought"]
    .count().reset_index()
)
test_df = (
    test_raw.groupby(["customer_id", "product_code"])["bought"]
    .count().reset_index()
)

# Drop test entries whose customer or item is not in train
train_customers = set(train_df["customer_id"].unique())
train_items     = set(train_df["product_code"].unique())
test_df = test_df[
    test_df["customer_id"].isin(train_customers) &
    test_df["product_code"].isin(train_items)
].copy()

# ── Integer encoding ─────────────────────────────────────────────────────────
customer_id2idx = {c: i for i, c in enumerate(train_df["customer_id"].unique())}
item_id2idx     = {a: i for i, a in enumerate(train_df["product_code"].unique())}

for df_ in [train_df, test_df]:
    df_["customer_id"]  = df_["customer_id"].map(customer_id2idx)
    df_["product_code"] = df_["product_code"].map(item_id2idx)
    df_.dropna(inplace=True)  # remove unmapped test entries
    df_[["customer_id", "product_code"]] = df_[["customer_id", "product_code"]].astype(int)

N_USERS = train_df["customer_id"].nunique()
N_ITEMS = train_df["product_code"].nunique()

print(f"Users: {N_USERS} | Items: {N_ITEMS}")
print(f"Train interactions: {len(train_df)} | Test interactions: {len(test_df)}")

In [ ]:
# ── Validation split from train (80/20 by row, random — mirrors ML-100K)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 2. Dataset & DataLoader

In [ ]:
class UserRandomSamplingDataset(Dataset):
    """
    Per-user dataset: each __getitem__ returns a random batch of that user's
    interactions.  Mirrors the ML-100K DataLoader design.
    """
    def __init__(self, df: pd.DataFrame, batch_size: int = 10):
        self.batch_size = batch_size
        # Build user -> indices lookup once
        users = torch.tensor(df["customer_id"].values, dtype=torch.long)
        items = torch.tensor(df["product_code"].values, dtype=torch.long)
        targets = torch.tensor(df["bought"].values, dtype=torch.float32)

        unique_users = users.unique()
        self.user_ids = unique_users
        self.user_to_indices = {
            u.item(): (users == u).nonzero(as_tuple=True)[0]
            for u in unique_users
        }
        self.items   = items
        self.targets = targets

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid = self.user_ids[idx].item()
        indices = self.user_to_indices[uid]
        bs = min(len(indices), self.batch_size)
        sel = indices[torch.randperm(len(indices))[:bs]]
        # Return (user_idx, item_idx) pairs and targets
        x = torch.stack([
            torch.full((bs,), uid, dtype=torch.long),
            self.items[sel]
        ], dim=1)  # shape: (bs, 2)  col0=user, col1=item
        y = self.targets[sel]  # shape: (bs,)
        return x, y


BATCH_SIZE = 10

train_dataset = UserRandomSamplingDataset(train_df, batch_size=BATCH_SIZE)
val_dataset   = UserRandomSamplingDataset(val_df,   batch_size=BATCH_SIZE)
test_dataset  = UserRandomSamplingDataset(test_df,  batch_size=BATCH_SIZE)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

print(f"Train loader batches: {len(train_loader)} (= N_USERS = {N_USERS})")

## 3. MF Model

In [ ]:
class MatrixFactorization(nn.Module):
    """Standard MF: yhat_ij = p_i . q_j, with L2-regularised embeddings."""
    def __init__(self, n_users: int, n_items: int, rank: int = 30, lam: float = 0.01):
        super().__init__()
        self.P = nn.Embedding(n_users, rank)  # user embeddings — local, never shared
        self.Q = nn.Embedding(n_items, rank)  # item embeddings — gradients shared
        self.lam = lam
        nn.init.normal_(self.P.weight, std=0.01)
        nn.init.normal_(self.Q.weight, std=0.01)

    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        p = self.P(user_ids)   # (bs, rank)
        q = self.Q(item_ids)   # (bs, rank)
        return (p * q).sum(dim=-1)  # (bs,)

    def l2_reg(self, user_ids: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        return self.lam * (
            self.P(user_ids).norm(dim=-1).pow(2).mean() +
            self.Q(item_ids).norm(dim=-1).pow(2).mean()
        )


@dataclass
class UserNode:
    user_id: int
    model: MatrixFactorization
    optimizer: torch.optim.Optimizer


def make_user_models(
    n_users: int, n_items: int,
    rank: int = 30, lam: float = 0.01, lr: float = 0.005
) -> Dict[int, UserNode]:
    """
    Create one MF model per user.  All share the SAME initial Q weights
    (item embeddings) — this is the shared population parameter.
    """
    # Shared initial Q
    shared_Q = nn.Embedding(n_items, rank)
    nn.init.normal_(shared_Q.weight, std=0.01)

    user_models = {}
    for uid in range(n_users):
        model = MatrixFactorization(n_users, n_items, rank=rank, lam=lam)
        # Copy shared Q into every user's model
        model.Q.weight.data.copy_(shared_Q.weight.data)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
        user_models[uid] = UserNode(user_id=uid, model=model, optimizer=optimizer)
    return user_models

print("MF model defined.")

## 4. Graph Topologies

In [ ]:
def build_graph(topology: str, n_nodes: int, seed: int = 42) -> nx.DiGraph:
    """Build directed communication graphs used in the paper."""
    rng = np.random.default_rng(seed)

    if topology == "random_2_out":
        G = nx.random_k_out_graph(n_nodes, k=2, alpha=0.5, self_loops=False, seed=seed)

    elif topology == "random_5_out":
        G = nx.random_k_out_graph(n_nodes, k=5, alpha=0.5, self_loops=False, seed=seed)

    elif topology == "scale_free":
        G = nx.scale_free_graph(n_nodes, alpha=0.5, beta=0.25, gamma=0.25, seed=seed)
        G = nx.DiGraph(G)  # remove parallel edges
        G.remove_edges_from(nx.selfloop_edges(G))

    elif topology == "cycle":
        # Directed cycle: i -> i+1 (mod n)
        G = nx.DiGraph()
        G.add_nodes_from(range(n_nodes))
        G.add_edges_from([(i, (i + 1) % n_nodes) for i in range(n_nodes)])

    else:
        raise ValueError(f"Unknown topology: {topology}")

    return G


TOPOLOGIES = ["random_2_out", "random_5_out", "scale_free", "cycle"]
TOPOLOGY_LABELS = {
    "random_2_out": "Random-2-out",
    "random_5_out": "Random-5-out",
    "scale_free":   "Scale-Free",
    "cycle":        "Cycle",
}

print("Graphs defined:", TOPOLOGIES)

## 5. FOAF Gradient Sharing

In [ ]:
BYTES_PER_PARAM = 4  # float32

def share_gradient(
    sender: UserNode,
    user_models: Dict[int, UserNode],
    graph: nx.DiGraph,
    item_ids: Optional[torch.Tensor] = None,
):
    """
    FOAF gradient sharing (Algorithm A5):
    - Sender's ∇Q for the items in the current batch is pushed to all
      out-neighbours in the graph.
    - Neighbours immediately apply the received gradient to their Q.
    - User embeddings (P) are NEVER shared.

    Returns:
        n_commutes : number of neighbour updates performed
        cost_bytes : bytes transmitted
    """
    uid = sender.user_id
    neighbors = list(graph.successors(uid))
    if len(neighbors) == 0:
        return 0, 0.0

    Q_grad = sender.model.Q.weight.grad  # shape: (n_items, rank)
    if Q_grad is None:
        return 0, 0.0

    if item_ids is not None:
        # Sparse share: only the rows touched in this batch
        unique_items = item_ids.unique()
        grad_to_share = Q_grad[unique_items]          # (k, rank)
        n_params = grad_to_share.numel()
        for nid in neighbors:
            neighbor = user_models[nid]
            with torch.no_grad():
                neighbor.model.Q.weight.data[unique_items] -= (
                    neighbor.optimizer.param_groups[0]["lr"] * grad_to_share
                )
    else:
        # Dense share: entire Q gradient
        grad_to_share = Q_grad
        n_params = grad_to_share.numel()
        for nid in neighbors:
            neighbor = user_models[nid]
            with torch.no_grad():
                neighbor.model.Q.weight.data -= (
                    neighbor.optimizer.param_groups[0]["lr"] * grad_to_share
                )

    cost_bytes = n_params * BYTES_PER_PARAM * len(neighbors)
    return len(neighbors), cost_bytes

print("share_gradient defined.")

## 6. Train / Evaluate Loop

In [ ]:
def evaluate_rmse(user_models: Dict[int, UserNode], loader: DataLoader) -> float:
    """Compute RMSE across all (user, item) pairs in the loader."""
    for _, node in user_models.items():
        node.model.eval()
    sq_errors, n_obs = 0.0, 0
    with torch.no_grad():
        for inputs, target in loader:
            if inputs.ndim == 3:
                inputs, target = inputs.squeeze(0), target.squeeze(0)
            uid = int(inputs[0, 0])
            score = user_models[uid].model(
                inputs[:, 0], inputs[:, 1]
            )
            sq_errors += ((score - target.float()) ** 2).sum().item()
            n_obs += inputs.shape[0]
    return math.sqrt(sq_errors / max(n_obs, 1))


def train_one_epoch(
    user_models: Dict[int, UserNode],
    loader: DataLoader,
    graph: nx.DiGraph,
    progress_bar: bool = False,
):
    """
    One training epoch following FOAF Algorithm 1:
      for each user i (random order):
        1. compute loss & gradients on local batch
        2. update own P and Q
        3. share ∇Q with graph neighbours
    Returns avg training RMSE, total commutes, total bytes.
    """
    loss_fn = nn.MSELoss(reduction="mean")
    for _, node in user_models.items():
        node.model.train()

    total_sq, total_n = 0.0, 0
    total_commutes, total_bytes = 0, 0.0

    it = tqdm(loader, leave=False) if progress_bar else loader
    for inputs, target in it:
        if inputs.ndim == 3:
            inputs, target = inputs.squeeze(0), target.squeeze(0)

        uid     = int(inputs[0, 0])
        item_ids = inputs[:, 1]             # (bs,)
        node    = user_models[uid]

        node.optimizer.zero_grad()
        score = node.model(inputs[:, 0], item_ids)
        loss  = loss_fn(score, target.float())
        loss += node.model.l2_reg(inputs[:, 0], item_ids)
        loss.backward()

        # Update own parameters FIRST
        node.optimizer.step()

        # THEN share ∇Q with neighbours (sparse — only items in batch)
        n_comm, n_bytes = share_gradient(
            node, user_models, graph, item_ids=item_ids
        )
        total_commutes += n_comm
        total_bytes    += n_bytes

        with torch.no_grad():
            bs = inputs.shape[0]
            total_sq += ((score.detach() - target.float()) ** 2).sum().item()
            total_n  += bs

    train_rmse = math.sqrt(total_sq / max(total_n, 1))
    return train_rmse, total_commutes, total_bytes


print("Training functions defined.")

## 7. Full Experiment Loop — FOAF across Topologies

In [ ]:
# ── Hyperparameters (mirror ML-100K notebook) ────────────────────────────────
RANK       = 30
LR         = 0.005
LAM        = 0.01
MAX_EPOCHS = 50
PATIENCE   = 5      # early stopping patience

# ── Results storage ──────────────────────────────────────────────────────────
results = {}   # topology -> {train_rmse, val_rmse, test_rmse, comm_bytes, epochs}
history = {}   # topology -> list of (epoch, train_rmse, val_rmse, cum_bytes)

for topo in TOPOLOGIES:
    print(f"\n{'='*60}")
    print(f"  Topology: {TOPOLOGY_LABELS[topo]}")
    print(f"{'='*60}")

    graph = build_graph(topo, N_USERS, seed=42)
    user_models = make_user_models(N_USERS, N_ITEMS, rank=RANK, lam=LAM, lr=LR)

    best_val_rmse = float("inf")
    patience_counter = 0
    best_state = None
    topo_history = []
    cum_bytes = 0.0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_rmse, n_comm, n_bytes = train_one_epoch(
            user_models, train_loader, graph, progress_bar=False
        )
        val_rmse  = evaluate_rmse(user_models, val_loader)
        cum_bytes += n_bytes

        topo_history.append({
            "epoch":      epoch,
            "train_rmse": train_rmse,
            "val_rmse":   val_rmse,
            "cum_mb":     cum_bytes / 1e6,
        })

        print(f"  Epoch {epoch:3d} | Train RMSE: {train_rmse:.4f} "
              f"| Val RMSE: {val_rmse:.4f} "
              f"| Cumulative comm: {cum_bytes/1e6:.1f} MB")

        # Early stopping
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy({
                uid: copy.deepcopy(node.model.state_dict())
                for uid, node in user_models.items()
            })
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}.")
                break

    # Restore best models and evaluate on test set
    for uid, node in user_models.items():
        node.model.load_state_dict(best_state[uid])

    test_rmse = evaluate_rmse(user_models, test_loader)
    print(f"\n  >> Best Val RMSE: {best_val_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

    results[topo] = {
        "best_val_rmse": best_val_rmse,
        "test_rmse":     test_rmse,
        "total_mb":      cum_bytes / 1e6,
        "epochs":        epoch,
    }
    history[topo] = topo_history

print("\n\nAll topologies complete.")

## 8. Results Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Graph":        TOPOLOGY_LABELS[t],
        "Epochs":       results[t]["epochs"],
        "Val RMSE":     round(results[t]["best_val_rmse"], 4),
        "Test RMSE":    round(results[t]["test_rmse"], 4),
        "Comm (MB)":    round(results[t]["total_mb"], 2),
    }
    for t in TOPOLOGIES
])
print(summary.to_string(index=False))

## 9. Figures — Val RMSE per Epoch & Comm vs RMSE

In [ ]:
COLORS = {
    "random_2_out": "#1f77b4",
    "random_5_out": "#ff7f0e",
    "scale_free":   "#2ca02c",
    "cycle":        "#d62728",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Val RMSE per epoch ─────────────────────────────────────────────────
ax = axes[0]
for topo in TOPOLOGIES:
    hist = history[topo]
    epochs   = [h["epoch"]    for h in hist]
    val_rmse = [h["val_rmse"] for h in hist]
    ax.plot(epochs, val_rmse,
            label=TOPOLOGY_LABELS[topo],
            color=COLORS[topo], linewidth=1.8)
    ax.scatter(epochs[-1], val_rmse[-1],
               color=COLORS[topo], s=60, zorder=5)

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Validation RMSE", fontsize=12)
ax.set_title("H&M — Val RMSE per Epoch (FOAF)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Right: Cumulative comm cost vs Val RMSE ──────────────────────────────────
ax = axes[1]
for topo in TOPOLOGIES:
    hist = history[topo]
    cum_mb   = [h["cum_mb"]   for h in hist]
    val_rmse = [h["val_rmse"] for h in hist]
    sizes    = [20 + 8 * h["epoch"] for h in hist]  # grow with epoch
    ax.plot(cum_mb, val_rmse,
            label=TOPOLOGY_LABELS[topo],
            color=COLORS[topo], linewidth=1.6)
    ax.scatter(cum_mb, val_rmse,
               s=sizes, color=COLORS[topo], alpha=0.6, zorder=5)

ax.set_xlabel("Cumulative Communication Cost (MB)", fontsize=12)
ax.set_ylabel("Validation RMSE", fontsize=12)
ax.set_title("H&M — Comm Cost vs Val RMSE (FOAF)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/hm_foaf_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to figures/hm_foaf_results.png")

## 10. Baselines

For a complete comparison table matching Table 2 in the appendix,
run these three baselines.

### Baseline definitions
- **LL (Local Learning)**: no gradient sharing — each user trains on their data only
- **CL (Centralized)**: all data pooled, single MF model trained centrally
- **GL (Gossip Learning)**: random-pair model averaging per round
- **FL (FedAvg)**: central server averages Q across all users each epoch

In [ ]:
# ── Local Learning (LL) ──────────────────────────────────────────────────────
def run_local_learning(n_users, n_items, rank=RANK, lam=LAM, lr=LR,
                       max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Each user trains independently — zero communication."""
    # Dummy disconnected graph (no edges)
    null_graph = nx.DiGraph()
    null_graph.add_nodes_from(range(n_users))

    user_models = make_user_models(n_users, n_items, rank=rank, lam=lam, lr=lr)
    best_val, best_state = float("inf"), None
    p_count = 0

    for epoch in range(1, max_epochs + 1):
        train_one_epoch(user_models, train_loader, null_graph)
        val_rmse = evaluate_rmse(user_models, val_loader)
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {uid: copy.deepcopy(n.model.state_dict())
                          for uid, n in user_models.items()}
            p_count = 0
        else:
            p_count += 1
            if p_count >= patience:
                break

    for uid, node in user_models.items():
        node.model.load_state_dict(best_state[uid])
    test_rmse = evaluate_rmse(user_models, test_loader)
    print(f"LL  | Val RMSE: {best_val:.4f} | Test RMSE: {test_rmse:.4f}")
    return test_rmse


# ── Centralized Learning (CL) ────────────────────────────────────────────────
def run_centralized(n_users, n_items, rank=RANK, lam=LAM, lr=LR,
                    max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """All data pooled; single shared MF model."""
    model = MatrixFactorization(n_users, n_items, rank=rank, lam=lam)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss(reduction="mean")

    # Wrap as a single-user system (uid always 0, remap user ids)
    # Use full train_df directly
    best_val, best_state, p_count = float("inf"), None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        idx = torch.randperm(len(train_df))
        u_t = torch.tensor(train_df["customer_id"].values[idx], dtype=torch.long)
        i_t = torch.tensor(train_df["product_code"].values[idx], dtype=torch.long)
        y_t = torch.tensor(train_df["bought"].values[idx],    dtype=torch.float32)

        # Mini-batch SGD with batch size 256
        for start in range(0, len(u_t), 256):
            ub = u_t[start:start+256]
            ib = i_t[start:start+256]
            yb = y_t[start:start+256]
            optimizer.zero_grad()
            score = model(ub, ib)
            loss  = loss_fn(score, yb) + model.l2_reg(ub, ib)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            v_u = torch.tensor(val_df["customer_id"].values, dtype=torch.long)
            v_i = torch.tensor(val_df["product_code"].values, dtype=torch.long)
            v_y = torch.tensor(val_df["bought"].values, dtype=torch.float32)
            pred = model(v_u, v_i)
            val_rmse = math.sqrt(((pred - v_y) ** 2).mean().item())

        if val_rmse < best_val:
            best_val = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            p_count = 0
        else:
            p_count += 1
            if p_count >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        t_u = torch.tensor(test_df["customer_id"].values, dtype=torch.long)
        t_i = torch.tensor(test_df["product_code"].values, dtype=torch.long)
        t_y = torch.tensor(test_df["bought"].values, dtype=torch.float32)
        pred = model(t_u, t_i)
        test_rmse = math.sqrt(((pred - t_y) ** 2).mean().item())

    print(f"CL  | Val RMSE: {best_val:.4f} | Test RMSE: {test_rmse:.4f}")
    return test_rmse


# ── FedAvg (FL) ──────────────────────────────────────────────────────────────
def run_fedavg(n_users, n_items, rank=RANK, lam=LAM, lr=LR,
               max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """
    FedAvg: each epoch, every user trains locally, then the server
    averages all Q weights and broadcasts back.
    """
    user_models = make_user_models(n_users, n_items, rank=rank, lam=lam, lr=lr)
    null_graph  = nx.DiGraph()
    null_graph.add_nodes_from(range(n_users))
    loss_fn = nn.MSELoss(reduction="mean")

    best_val, best_state, p_count = float("inf"), None, 0

    for epoch in range(1, max_epochs + 1):
        # Local training step (no sharing)
        for _, node in user_models.items():
            node.model.train()
        for inputs, target in train_loader:
            if inputs.ndim == 3:
                inputs, target = inputs.squeeze(0), target.squeeze(0)
            uid  = int(inputs[0, 0])
            node = user_models[uid]
            node.optimizer.zero_grad()
            score = node.model(inputs[:, 0], inputs[:, 1])
            loss  = loss_fn(score, target.float())
            loss += node.model.l2_reg(inputs[:, 0], inputs[:, 1])
            loss.backward()
            node.optimizer.step()

        # FedAvg aggregation: average Q across all users
        avg_Q = torch.zeros(n_items, rank)
        for _, node in user_models.items():
            avg_Q += node.model.Q.weight.data
        avg_Q /= n_users
        for _, node in user_models.items():
            node.model.Q.weight.data.copy_(avg_Q)

        val_rmse = evaluate_rmse(user_models, val_loader)
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {uid: copy.deepcopy(n.model.state_dict())
                          for uid, n in user_models.items()}
            p_count = 0
        else:
            p_count += 1
            if p_count >= patience:
                break

    for uid, node in user_models.items():
        node.model.load_state_dict(best_state[uid])
    test_rmse = evaluate_rmse(user_models, test_loader)
    print(f"FL  | Val RMSE: {best_val:.4f} | Test RMSE: {test_rmse:.4f}")
    return test_rmse


# ── Gossip Learning (GL) ─────────────────────────────────────────────────────
def run_gossip(n_users, n_items, rank=RANK, lam=LAM, lr=LR,
               max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """
    GL: after local update, each user picks one random neighbour and
    averages Q weights with them.
    """
    graph = build_graph("random_2_out", n_users)  # use r-2-out as the gossip graph
    user_models = make_user_models(n_users, n_items, rank=rank, lam=lam, lr=lr)
    loss_fn = nn.MSELoss(reduction="mean")

    best_val, best_state, p_count = float("inf"), None, 0

    for epoch in range(1, max_epochs + 1):
        for _, node in user_models.items():
            node.model.train()
        for inputs, target in train_loader:
            if inputs.ndim == 3:
                inputs, target = inputs.squeeze(0), target.squeeze(0)
            uid  = int(inputs[0, 0])
            node = user_models[uid]
            node.optimizer.zero_grad()
            score = node.model(inputs[:, 0], inputs[:, 1])
            loss  = loss_fn(score, target.float())
            loss += node.model.l2_reg(inputs[:, 0], inputs[:, 1])
            loss.backward()
            node.optimizer.step()

            # Gossip: average Q with a random neighbour
            neighbors = list(graph.successors(uid))
            if neighbors:
                nid = int(np.random.choice(neighbors))
                with torch.no_grad():
                    avg_Q = (node.model.Q.weight.data +
                             user_models[nid].model.Q.weight.data) / 2
                    node.model.Q.weight.data.copy_(avg_Q)
                    user_models[nid].model.Q.weight.data.copy_(avg_Q)

        val_rmse = evaluate_rmse(user_models, val_loader)
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {uid: copy.deepcopy(n.model.state_dict())
                          for uid, n in user_models.items()}
            p_count = 0
        else:
            p_count += 1
            if p_count >= patience:
                break

    for uid, node in user_models.items():
        node.model.load_state_dict(best_state[uid])
    test_rmse = evaluate_rmse(user_models, test_loader)
    print(f"GL  | Val RMSE: {best_val:.4f} | Test RMSE: {test_rmse:.4f}")
    return test_rmse


print("Baseline functions defined. Run the next cell to execute them.")

In [ ]:
# ── Run all baselines ─────────────────────────────────────────────────────────
# Note: CL and LL are topology-independent; GL and FL reported for r-2-out.
print("Running baselines...")
ll_rmse = run_local_learning(N_USERS, N_ITEMS)
cl_rmse = run_centralized(N_USERS, N_ITEMS)
fl_rmse = run_fedavg(N_USERS, N_ITEMS)
gl_rmse = run_gossip(N_USERS, N_ITEMS)

## 11. Final Comparison Table (mirrors Table 2 in appendix)

In [ ]:
print("\n" + "="*70)
print("  H&M Dataset — Test RMSE Comparison")
print("="*70)
print(f"  {'Graph':<20} {'FOAF':>10} {'CL':>8} {'FL':>8} {'GL':>8} {'LL':>8}")
print("-"*70)
for topo in TOPOLOGIES:
    foaf = results[topo]["test_rmse"]
    print(f"  {TOPOLOGY_LABELS[topo]:<20} {foaf:>10.4f} "
          f"{cl_rmse:>8.4f} {fl_rmse:>8.4f} "
          f"{gl_rmse:>8.4f} {ll_rmse:>8.4f}")
print("="*70)

# ── Save CSVs for LaTeX table generation ─────────────────────────────────────
rows = []
for topo in TOPOLOGIES:
    rows.append({
        "graph":    TOPOLOGY_LABELS[topo],
        "foaf":     results[topo]["test_rmse"],
        "cl":       cl_rmse,
        "fl":       fl_rmse,
        "gl":       gl_rmse,
        "ll":       ll_rmse,
        "comm_mb":  results[topo]["total_mb"],
        "epochs":   results[topo]["epochs"],
    })
comparison_df = pd.DataFrame(rows)
os.makedirs("results", exist_ok=True)
comparison_df.to_csv("results/hm_test_rmse_comparison.csv", index=False)

# Val RMSE per epoch per topology
for topo in TOPOLOGIES:
    pd.DataFrame(history[topo]).to_csv(
        f"results/hm_history_{topo}.csv", index=False
    )

print("\nResults saved to results/hm_test_rmse_comparison.csv")